Task 1: Conceptual Understanding

1. Difference between "Love" and "love"In NLP, these are two distinct tokens because computers use ASCII/Unicode values (where 'L' is 76 and 'l' is 108). Without case normalization, a model treats them as different words, splitting the statistical weight of the term.
 
 2. Impact of not removing StopwordsStopwords (is, the, a, at) are the most frequent words in a language but carry the least "signal." If they aren't removed, they can dominate frequency analysis and increase the dimensionality of the dataset, making models slower and less accurate.
 
 3. When is removing Stopwords harmful?
 Sentiment Analysis: Removing "not" from "I am not happy" changes the meaning entirely.Contextual Phrases: In a phrase like "To be or not to be," every word is a stopword. Removing them leaves an empty string.
 
 4. Stemming vs. Lemmatization: 
 Stemming: A crude heuristic that chops off the ends of words (e.g., "studies" $\rightarrow$ "studi"). It is fast but often results in non-dictionary words.
 Lemmatization: Uses vocabulary and morphological analysis to return a word to its dictionary base form (e.g., "studies" $\rightarrow$ "study"). It is more accurate but computationally more expensive.

In [14]:
import re
from collections import Counter

def preprocess_text(text):
    # Task 7: Error Handling (Empty strings, Non-strings, or Only Numbers)
    if not text or not isinstance(text, str):
        return [], ""

    # Convert to lowercase
    text = text.lower()

    # Remove URLs and Email-like patterns
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'\S+@\S+', '', text)

    # Remove numbers
    text = re.sub(r'\d+', '', text)

    # Handle repeated characters (e.g., "soooo goooood!!!" -> "so good")
    # Reduces 3+ repetitions to 1 to clean slang noise
    text = re.sub(r'(.)\1{2,}', r'\1', text)

    # Remove special characters/emojis (Keeping only a-z and spaces)
    text = re.sub(r'[^a-z\s]', '', text)
    
    # Tokenize and remove extra spaces
    raw_tokens = text.split()

    # Remove short tokens (len <= 2) EXCEPT meaningful words like "no", "not"
    keep_words = {'no', 'not'}
    tokens = [t for t in raw_tokens if len(t) > 2 or t in keep_words]

    cleaned_sentence = " ".join(tokens)
    return tokens, cleaned_sentence

In [15]:
sample_inputs = [
    "Get 100% FREE access now!!!",
    "I absolutely looooved this product",
    "Worst service ever... 0/10",
    "Call me at 9876543210",
    "This is THE best course!!!",
    "Visit https://openai.com now!",
    "Nooooo this is baaad!!!",
    "OK OK OK I got it",
    "Win $$$ now!!! Limited offer!!!",
    "I am not happy with this"
]

all_processed_tokens = []

print(f"{'Original Text':<35} | {'Cleaned Sentence':<25} | {'Avg Len'}")
print("-" * 85)

for s in sample_inputs:
    tokens, clean_sent = preprocess_text(s)
    all_processed_tokens.extend(tokens)
    
    # Task 4: Analytics
    avg_len = sum(len(t) for t in tokens) / len(tokens) if tokens else 0
    print(f"{s[:35]:<40} | {clean_sent[:40]:<40} | {avg_len:.2f}")

# Task 4: Analysis Answers
# Q: Which sentence had the most noise? 
# A: "Visit https://openai.com now!" (Due to URL and punctuation)
# Q: Which sentence retained the most meaningful tokens?
# A: "I absolutely looooved this product" (Core tokens 'absolutely', 'loved', 'product' survived)

Original Text                       | Cleaned Sentence          | Avg Len
-------------------------------------------------------------------------------------
Get 100% FREE access now!!!              | get free access now                      | 4.00
I absolutely looooved this product       | absolutely loved this product            | 6.50
Worst service ever... 0/10               | worst service ever                       | 5.33
Call me at 9876543210                    | call                                     | 4.00
This is THE best course!!!               | this the best course                     | 4.25
Visit https://openai.com now!            | visit now                                | 4.00
Nooooo this is baaad!!!                  | no this bad                              | 3.00
OK OK OK I got it                        | got                                      | 3.00
Win $$$ now!!! Limited offer!!!          | win now limited offer                    | 4.50
I am not happy with t

In [16]:
# Task 5: Frequency Analysis
word_counts = Counter(all_processed_tokens)
print("\nTop 10 Most Frequent Words:", word_counts.most_common(10))
print("Top 5 Least Frequent Words:", word_counts.most_common()[:-6:-1])

# Task 6: Full Pipeline
def full_pipeline(text_list):
    results = {"tokens": [], "clean_sentences": []}
    for text in text_list:
        tokens, clean_sent = preprocess_text(text)
        results["tokens"].append(tokens)
        results["clean_sentences"].append(clean_sent)
    return results

# Final Pipeline verification
final_results = full_pipeline(sample_inputs)
print("\nFull Pipeline Verification (First 2 entries):")
print(final_results["clean_sentences"][:2])


Top 10 Most Frequent Words: [('this', 4), ('now', 3), ('get', 1), ('free', 1), ('access', 1), ('absolutely', 1), ('loved', 1), ('product', 1), ('worst', 1), ('service', 1)]
Top 5 Least Frequent Words: [('with', 1), ('happy', 1), ('not', 1), ('offer', 1), ('limited', 1)]

Full Pipeline Verification (First 2 entries):
['get free access now', 'absolutely loved this product']
